[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/prompts/phoenix_prompt_tutorial.ipynb)


# Phoenix 提示词教程配套 Notebook

本 Notebook 配合 Phoenix 提示词快速入门文档使用。请结合文档中的详细说明操作。

**准备条件：**
- 在本地运行 Phoenix（`phoenix serve`）
- 将 OpenAI API 密钥设置为 `OPENAI_API_KEY` 环境变量


In [ ]:
import os
from getpass import getpass

# 如果尚未配置，则设置 OpenAI API 密钥
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")

---
# 第一部分：查找并编辑提示词

## 第一步：在追踪中定位表现不佳的 Span

先构建一个客服 Agent 并接入追踪，生成一些可供检查的追踪记录。


In [ ]:
# 定义分类任务的系统提示词
system_prompt = """
Account Creation
Login Issues
Password Reset
Two-Factor Authentication
Profile Updates
Billing Inquiry
Refund Request
Subscription Upgrade/Downgrade
Payment Method Update
Invoice Request
Order Status
Shipping Delay
Product Return
Warranty Claim
Technical Bug Report
Feature Request
Integration Help
Data Export
Security Concern
Terms of Service Question
Privacy Policy Question
Compliance Inquiry
Accessibility Support
Language Support
Mobile App Issue
Desktop App Issue
Email Notifications
Marketing Preferences
Beta Program Enrollment
General Feedback

针对客服问题，仅返回上述英文分类名称，不要输出其他文本。
"""

In [ ]:
import json

import requests

url = "https://storage.googleapis.com/arize-phoenix-assets/assets/images/guidelines.json"
response = requests.get(url)

with open("guidelines.json", "wb") as f:
    f.write(response.content)

with open("guidelines.json", "r") as f:
    guidelines = json.load(f)

print("✅ 已加载 guidelines.json")

In [ ]:
# 构建客服 Agent 并接入追踪
from openai import OpenAI
from phoenix.otel import register

# 配置 Phoenix 追踪，并为 OpenAI 启用自动埋点
tracer_provider = register(project_name="support-agent", auto_instrument=True)
tracer = tracer_provider.get_tracer(__name__)

client = OpenAI()


@tracer.tool
def retrieve_guidelines(classification: str) -> str:
    """根据客服问题的分类检索处理指南。"""
    return guidelines.get(classification, "未找到处理指南。")


@tracer.chain
def handle_support_query(query: str) -> str:
    # 第一步：对问题分类
    classification_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ],
    )
    classification = classification_response.choices[0].message.content

    # 第二步：根据分类检索指南
    guideline = retrieve_guidelines(classification)

    # 第三步：根据指南生成最终回复
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"根据以下指南回答客服问题：\n{guideline}",
            },
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content

In [ ]:
# 对测试问题运行 Agent，生成追踪
queries = [
    "warranty reg page says 404",
    "every time i click settings, bye",
    "when's dark mode? u said soon",
    "calendar sync eats my events",
    "cant dl my info, button grayed",
]

for query in queries:
    print(f"问题：{query}")
    result = handle_support_query(query)
    print(f"回答：{result[:200]}...")
    print("-" * 50)

## 第二步：在试验场中重放 Span 并编辑提示词

打开 Phoenix 界面并进入追踪页面。点击一个 Span，使用**试验场（Playground）**：
1. 将原始提示词保存为 `support-classifier`
2. 编辑提示词并测试修改效果
3. 将编辑结果保存为第 2 版

## 第三步：将编辑后的提示词加载回代码


In [ ]:
from phoenix.client import Client

px_client = Client()

# 拉取最新版本
prompt = px_client.prompts.get(prompt_identifier="support-classifier")

# 也可以拉取指定版本
# prompt = px_client.prompts.get(prompt_version_id="YOUR_VERSION_ID")

print(f"已加载提示词：{prompt._model_name}")

---
# 第二部分：批量测试提示词

## 第一步：加载输入数据集


In [ ]:
import pandas as pd
from phoenix.client import Client

px_client = Client()

# 加载客服问题数据集
support_query_csv_url = (
    "https://storage.googleapis.com/arize-phoenix-assets/assets/images/support_queries.csv"
)
support_query_df = pd.read_csv(support_query_csv_url)

print(f"已加载 {len(support_query_df)} 个样本")
print(support_query_df.head())

In [ ]:
# 将数据集上传到 Phoenix
support_query_dataset = px_client.datasets.create_dataset(
    dataframe=support_query_df,
    name="support-query-dataset",
    input_keys=["query"],
    output_keys=["ground_truth"],
)

print(f"✅ 已创建数据集：{support_query_dataset.id}")

## 第二步：使用当前提示词运行实验

### 定义任务函数


In [ ]:
from openai import AsyncOpenAI
from phoenix.client import Client

async_openai_client = AsyncOpenAI()
px_client = Client()

prompt = px_client.prompts.get(prompt_identifier="support-classifier")
model = prompt._model_name
messages = prompt._template["messages"]

# 编辑用户提示词，使其匹配数据集输入键 "query"
messages[1]["content"][0]["text"] = "{{query}}"


async def task(input):
    task_messages = [
        {"role": m["role"], "content": [{"type": "text", "text": m["content"][0]["text"]}]}
        for m in messages
    ]
    task_messages[1]["content"][0]["text"] = input["query"]
    response = await async_openai_client.chat.completions.create(
        model=model,
        messages=task_messages,
    )
    return response.choices[0].message.content


print(f"已定义任务，使用模型：{model}")

### 定义评估器


In [ ]:
# 用于提供详细反馈的分析评估器模板
analysis_evaluator_template = """
你是客服工单分类评估专家。

任务：根据用户问题、模型预测分类和正确分类，判断预测是否正确，解释原因，分析可能的混淆原因，标出问题中最能支持正确分类的具体片段；若预测错误，还需标注错误类型。

可用英文分类如下：

Account Creation, Login Issues, Password Reset, Two-Factor Authentication, Profile Updates,
Billing Inquiry, Refund Request, Subscription Upgrade/Downgrade, Payment Method Update, Invoice Request,
Order Status, Shipping Delay, Product Return, Warranty Claim, Technical Bug Report, Feature Request,
Integration Help, Data Export, Security Concern, Terms of Service Question, Privacy Policy Question,
Compliance Inquiry, Accessibility Support, Language Support, Mobile App Issue, Desktop App Issue,
Email Notifications, Marketing Preferences, Beta Program Enrollment, General Feedback

---

**输入：**
- 用户问题：{query}
- 预测分类：{output}
- 正确分类：{ground_truth}

---

**错误类型定义：**
- **broad_vs_specific** → 模型选择了较宽泛的分类，而不是正确的具体分类，或反之。
- **keyword_bias** → 模型过度依赖孤立关键词，导致分类错误。
- **multi_intent_confusion** → 问题包含多种可能意图，模型选择了次要意图。
- **ambiguous_query** → 问题含糊或信息不足。
- **off_topic** → 问题与任何分类都不太匹配，但模型仍然猜测分类。
- **paraphrase_gap** → 模型未能识别以非标准表达描述的正确意图。
- **other** → 其他原因。
- **none** → 仅在 correctness 为 "correct" 时使用。

---

**输出格式（JSON；字段名与枚举值保持英文）：**
  "correctness": "correct" 或 "incorrect",
  "explanation": "简要解释预测分类正确或错误的原因。",
  "confusion_reason": "若错误，解释模型可能为何作出该选择；若正确，填 'no confusion'。",
  "error_type": "上述错误类型之一；若正确，填 'none'。",
  "evidence_span": "用户问题中明确支持正确分类的原始短语。",
  "prompt_fix_suggestion": "一条可加入分类提示词、用于避免该错误的明确指令。"
"""

In [ ]:
from phoenix.evals import create_evaluator
from phoenix.evals.llm import LLM

llm = LLM(provider="openai", model="gpt-4.1")


def normalize(label):
    return label.strip().strip('"').strip("'").lower()


async def ground_truth_evaluator(expected, output):
    """简单评估器：检查输出是否与真实标签一致。"""
    return normalize(expected.get("ground_truth")) == normalize(output)


SCHEMA = {
    "type": "object",
    "properties": {
        "correctness": {"type": "string", "enum": ["correct", "incorrect"]},
        "explanation": {"type": "string"},
        "confusion_reason": {"type": "string"},
        "error_type": {"type": "string"},
        "evidence_span": {"type": "string"},
        "prompt_fix_suggestion": {"type": "string"},
    },
    "required": [
        "correctness",
        "explanation",
        "confusion_reason",
        "error_type",
        "evidence_span",
        "prompt_fix_suggestion",
    ],
    "additionalProperties": False,
}


@create_evaluator(name="output_evaluator", kind="llm")
def analysis_evaluator(input, expected, output):
    """LLM 评估器：为分类错误提供详细反馈。"""
    query = input.get("query")
    ground_truth = expected.get("ground_truth")

    prompt = (
        analysis_evaluator_template.replace("{query}", query)
        .replace("{ground_truth}", ground_truth)
        .replace("{output}", output)
    )
    obj = llm.generate_object(prompt=prompt, schema=SCHEMA)
    correctness = obj["correctness"]
    score = 1.0 if correctness == "correct" else 0.0
    explanation = (
        f"correctness: {correctness}; "
        f"explanation: {obj.get('explanation', '')}; "
        f"confusion_reason: {obj.get('confusion_reason', '')}; "
        f"error_type: {obj.get('error_type', '')}; "
        f"evidence_span: {obj.get('evidence_span', '')}; "
        f"prompt_fix_suggestion: {obj.get('prompt_fix_suggestion', '')};"
    )
    return {"score": score, "label": correctness, "explanation": explanation}


print("✅ 已定义评估器")

### 运行实验


In [ ]:
from phoenix.client.experiments import async_run_experiment

experiment = await async_run_experiment(
    dataset=support_query_dataset,
    task=task,
    evaluators=[ground_truth_evaluator, analysis_evaluator],
    experiment_name="support-classifier-baseline",
)

## 第三步：分析实验结果

进入 Phoenix 界面查看实验结果。筛选分类错误的结果：
```
evals["output_evaluator"].score == 0
```

筛选 `broad_vs_specific` 类型的错误：
```
'broad_vs_specific' in evals["output_evaluator"].explanation
```


---
# 第三部分：比较提示词版本

## 编辑提示词模板（第 3 版）

根据分析结果，添加一条指令来处理 `broad_vs_specific` 类型的错误。


In [ ]:
from phoenix.client import Client
from phoenix.client.types.prompts import PromptVersion

px_client = Client()

# 添加处理 broad_vs_specific 错误的新指令
broad_vs_specific_instruction = """对用户问题分类时，始终优先选择最具体的适用分类，而非更宽泛的分类。如果问题提到明确、具体的操作或对象（例如订阅降级、发票、资料名称），应归入该具体意图，而不是 Billing Inquiry、General Feedback 等宽泛分类。"""

# 获取现有提示词
existing = px_client.prompts.get(prompt_identifier="support-classifier")

# 修改模板
messages = existing._template["messages"]
messages[0]["content"][0]["text"] += "\n\n" + broad_vs_specific_instruction

# 根据修改创建新版本
new_version = PromptVersion(
    messages,
    model_name=existing._model_name,
    model_provider=existing._model_provider,
    template_format=existing._template_format,
    description="添加 broad_vs_specific 规则",
)

# 保存为新版本
version_3 = px_client.prompts.create(
    name="support-classifier",
    version=new_version,
)

print(f"✅ 已创建第 3 版： {version_3.id}")

## 编辑提示词参数（第 4 版）

调整模型参数，创建另一个版本。


In [ ]:
# 获取现有提示词的新副本
existing = px_client.prompts.get(prompt_identifier="support-classifier")

new_version = PromptVersion(
    existing._template["messages"],
    model_name="gpt-4.1-mini",
    model_provider=existing._model_provider,
    template_format="MUSTACHE",
    description="使用 temperature=0.3、top_p=0.8、model_name=gpt-4.1-mini",
)

# 设置调用参数
new_version._invocation_parameters = {
    "temperature": 0.3,
    "top_p": 0.8,
}

version_4 = px_client.prompts.create(
    name="support-classifier",
    version=new_version,
)

print(f"✅ 已创建第 4 版： {version_4.id}")

## 比较提示词版本

从 Phoenix 界面复制版本 ID，再运行实验进行比较。


In [ ]:
from openai import AsyncOpenAI
from phoenix.client import Client
from phoenix.client.experiments import async_run_experiment

px_client = Client()
async_openai_client = AsyncOpenAI()

# 获取数据集
dataset = px_client.datasets.get_dataset(dataset="support-query-dataset")

# 版本 ID：替换为 Phoenix 界面中你自己的版本 ID
VERSION_3 = "REPLACE_WITH_VERSION_3_ID"
VERSION_4 = "REPLACE_WITH_VERSION_4_ID"

# 获取提示词版本
prompt_v3 = px_client.prompts.get(prompt_version_id=VERSION_3)
prompt_v4 = px_client.prompts.get(prompt_version_id=VERSION_4)

print(f"第 3 版模型： {prompt_v3._model_name}")
print(f"第 4 版模型： {prompt_v4._model_name}")

In [ ]:
# 定义可复用的任务工厂
def create_task(prompt):
    model = prompt._model_name
    messages = prompt._template["messages"].copy()

    async def task(input):
        # 创建副本，避免修改原对象
        task_messages = [
            {"role": m["role"], "content": [{"type": "text", "text": m["content"][0]["text"]}]}
            for m in messages
        ]
        task_messages[1]["content"][0]["text"] = input["query"]
        response = await async_openai_client.chat.completions.create(
            model=model,
            messages=task_messages,
        )
        return response.choices[0].message.content

    return task

In [ ]:
# 使用第 3 版运行实验
print("🧪 正在使用第 3 版运行实验……")
experiment_v3 = await async_run_experiment(
    dataset=dataset,
    task=create_task(prompt_v3),
    evaluators=[ground_truth_evaluator, analysis_evaluator],
    experiment_name="support-classifier-v3",
)

# 使用第 4 版运行实验
print("\n🧪 正在使用第 4 版运行实验……")
experiment_v4 = await async_run_experiment(
    dataset=dataset,
    task=create_task(prompt_v4),
    evaluators=[ground_truth_evaluator, analysis_evaluator],
    experiment_name="support-classifier-v4",
)

print(f"\n✅ 结果对比地址： http://localhost:6006/datasets/{dataset.id}/experiments")

---
# 第四部分：使用提示词学习进行优化

## 安装 Prompt Learning SDK

```bash
git clone https://github.com/priyanjindal/prompt-learning.git
cd prompt-learning
pip install .
```

## 加载实验用于训练


In [ ]:
import os

import pandas as pd
import requests


def process_experiment(experiment_id, feedback_columns=None):
    """
    从 Phoenix API 获取实验数据，并整理为 DataFrame。

    参数：
        experiment_id: Phoenix 实验 ID。
        feedback_columns: 要从标注中提取的反馈字段名列表。

    返回：
        pd.DataFrame: 整理后的实验数据。
    """
    url = f"{os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', 'http://localhost:6006')}/v1/experiments/{experiment_id}/json"
    headers = {}
    if os.environ.get("PHOENIX_API_KEY"):
        headers["Authorization"] = f"Bearer {os.environ['PHOENIX_API_KEY']}"

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise RuntimeError(
            f"获取实验数据失败： {response.status_code} {response.text}"
        )

    results = response.json()

    # 将实验结果整理为 DataFrame
    data = []
    for entry in results:
        row = {
            "input": entry.get("input", {}),
            "output": entry.get("output"),
            "ground_truth": entry.get("reference_output", {}).get("ground_truth"),
        }
        # 从输入中提取问题
        if isinstance(row["input"], dict):
            row["query"] = row["input"].get("query", "")

        # 从标注中提取反馈
        if feedback_columns and entry.get("annotations"):
            # 查找 output_evaluator 标注
            for annotation in entry["annotations"]:
                if annotation.get("name") == "output_evaluator":
                    eval_output = annotation.get("explanation", "")
                    for item in eval_output.split(";"):
                        if ":" in item:
                            key, value = item.split(":", 1)
                            key = key.strip()
                            if key in feedback_columns:
                                row[key] = value.strip()
                    break

        data.append(row)

    return pd.DataFrame(data)

In [ ]:
# 替换为 Phoenix 界面中第 4 版的实验 ID
EXPERIMENT_V4_ID = "REPLACE_WITH_EXPERIMENT_V4_ID"

# analysis_evaluator 提供的反馈列
feedback_columns = [
    "correctness",
    "explanation",
    "confusion_reason",
    "error_type",
    "evidence_span",
    "prompt_fix_suggestion",
]

processed_experiment_data = process_experiment(
    experiment_id=EXPERIMENT_V4_ID, feedback_columns=feedback_columns
)

print(f"已处理 {len(processed_experiment_data)} 行")
print(processed_experiment_data.head())

## 加载尚未优化的提示词


In [ ]:
import os

from phoenix.client import Client

px_client = Client()

# 替换为待优化提示词的版本 ID
PROMPT_VERSION_ID = "REPLACE_WITH_PROMPT_VERSION_ID"
unoptimized_prompt = px_client.prompts.get(prompt_version_id=PROMPT_VERSION_ID)

# 从 messages[0] 中提取系统提示词
system_prompt = unoptimized_prompt._template["messages"][0]["content"][0]["text"]

print(f"已加载系统提示词（{len(system_prompt)} 个字符）")

## 优化提示词（第 5 版）


In [ ]:
from prompt_learning import PromptLearningOptimizer

# 使用现有系统提示词初始化优化器
optimizer = PromptLearningOptimizer(
    prompt=system_prompt, model_choice="gpt-4o", openai_api_key=os.getenv("OPENAI_API_KEY")
)

# 运行优化
optimized_system_prompt = optimizer.optimize(
    dataset=processed_experiment_data,
    output_column="output",
    feedback_columns=feedback_columns,
    context_size_k=90000,
)

print("\n" + "=" * 60)
print("优化后的提示词")
print("=" * 60)
print(optimized_system_prompt[:500] + "...")

In [ ]:
from phoenix.client.types.prompts import PromptVersion

# 使用优化后的系统提示词构建消息
optimized_messages = [
    {"role": "system", "content": [{"type": "text", "text": optimized_system_prompt}]},
    {"role": "user", "content": [{"type": "text", "text": "{{query}}"}]},
]

# 使用优化后的提示词创建新版本
new_version = PromptVersion(
    optimized_messages,
    model_name=unoptimized_prompt._model_name,
    model_provider=unoptimized_prompt._model_provider,
    template_format="MUSTACHE",
    description="根据第 4 版实验使用 Prompt Learning 优化",
)

# 保留已有调用参数
if unoptimized_prompt._invocation_parameters:
    new_version._invocation_parameters = unoptimized_prompt._invocation_parameters

# 推送到 Phoenix
optimized_prompt = px_client.prompts.create(
    name="support-classifier",
    version=new_version,
)

print(f"✅ 已创建优化后的提示词版本： {optimized_prompt.id}")

## 衡量新版提示词的效果


In [ ]:
from phoenix.client.experiments import async_run_experiment

print("🧪 正在使用 Prompt Learning 优化后的提示词运行实验……")

experiment_optimized = await async_run_experiment(
    dataset=support_query_dataset,
    task=create_task(optimized_prompt),
    evaluators=[ground_truth_evaluator, analysis_evaluator],
    experiment_name="support-classifier-optimized",
)

print(f"\n✅ 实验已完成： {experiment_optimized.id}")
print("📊 查看结果： http://localhost:6006/experiments")

---
# 🎉 总结

你已完成 Phoenix 提示词教程！

**本教程介绍了如何：**
- 在 Phoenix 提示词中心**存储提示词并管理版本**
- 从 CSV 文件或 DataFrame **创建数据集并上传到 Phoenix**
- **构建自定义评估器**，包括基于代码和使用结构化输出的 LLM 评估器
- **运行实验**，批量测试提示词并自动记录评测结果
- 并排**比较提示词版本**，衡量修改的影响
- **使用 Prompt Learning 优化提示词**，依据实验反馈自动生成改进方案
